In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# !pip install pandas openpyxl scikit-learn torch -q

In [3]:
# %run mc_dropout_revised.py

Proper Monte Carlo Dropout experiment
for GA-ANN manuscript revision.

Important:
Dropout is included DURING TRAINING from the beginning.
The model is NOT created after training and is NOT initialized
from a previously trained no-dropout model.

Method:
- Raw 70/30 development/test split
- log1p suction
- StandardScaler fitted only on development training data
- GA-selected architecture: H1=54, H2=51
- GA-selected learning rate: 0.0098003061
- Dropout = 0.20
- Dropout-trained model
- 200 stochastic forward passes
- 95% empirical prediction intervals
- Coverage calculation
- Independent test metrics
- CP > 50% analysis

In [4]:
import os
import json
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

In [5]:
# COLAB CONFIGURATION

DATA_PATH = "/content/drive/MyDrive/PINNs/Suction_vsCP-modified_1.xlsx"

OUT_DIR = "/content/drive/MyDrive/NNsGA/mc_dropout_results"

SEED = 42

# GA-selected architecture from revised GA experiment
H1 = 54
H2 = 51

# GA-selected learning rate
LEARNING_RATE = 0.0098003061

# Dropout probability
DROPOUT_RATE = 0.20

# Monte Carlo samples
MC_RUNS = 200

# Training
EPOCHS = 500
BATCH_SIZE = 64

# Internal validation fraction from DEVELOPMENT set
VALIDATION_SIZE = 0.15

# Early stopping
PATIENCE = 50

os.makedirs(OUT_DIR, exist_ok=True)

In [6]:
# REPRODUCIBILITY

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 40)
print("PROPER MONTE CARLO DROPOUT EXPERIMENT")
print("=" * 40)

print(f"Device: {DEVICE}")
print(f"Architecture: {H1}-{H2}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Dropout rate: {DROPOUT_RATE}")
print(f"MC runs: {MC_RUNS}")

PROPER MONTE CARLO DROPOUT EXPERIMENT
Device: cuda
Architecture: 54-51
Learning rate: 0.0098003061
Dropout rate: 0.2
MC runs: 200


In [7]:
# COLUMNS
TARGET_COL = "Collapse Potential (%)"

FEATURE_COLS = [
    "Suction (kPa)",
    "Silica fume (%)",
    "Lime (%)",
    "Gypsum content (%)",
    "Applied vertical stress (kPa)",
    "Degree of Saturation (%)",
]

In [8]:
# LOAD DATA
df = pd.read_excel(DATA_PATH)
df = df[FEATURE_COLS + [TARGET_COL]].dropna().copy()
X = df[FEATURE_COLS].values.astype(np.float32)
y = df[TARGET_COL].values.astype(np.float32)
print(f"\nTotal observations: {len(df)}")


Total observations: 600


In [9]:
# RAW 70/30 SPLIT
X_dev, X_test, y_dev, y_test = train_test_split(
                                  X,
                                  y,
                                  test_size=0.30,
                                  random_state=SEED
                              )
print(f"Development set: {len(X_dev)}")
print(f"Independent test set: {len(X_test)}")


Development set: 420
Independent test set: 180


In [10]:
# INTERNAL TRAIN/VALIDATION SPLIT

X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=VALIDATION_SIZE,
    random_state=SEED
)

print(f"Training subset: {len(X_train)}")
print(f"Validation subset: {len(X_val)}")

Training subset: 357
Validation subset: 63


In [11]:
# LEAKAGE-FREE PREPROCESSING
def preprocess_fit(X_train):

    X_train = X_train.copy()

    # Suction is strongly right-skewed
    X_train[:, 0] = np.log1p(
        np.clip(X_train[:, 0], 0, None)
    )

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X_train)

    return scaler, X_scaled


def preprocess_transform(X_data, scaler):

    X_data = X_data.copy()

    X_data[:, 0] = np.log1p(
        np.clip(X_data[:, 0], 0, None)
    )

    return scaler.transform(X_data)


scaler, X_train_scaled = preprocess_fit(X_train)

X_val_scaled = preprocess_transform(
    X_val,
    scaler)

X_test_scaled = preprocess_transform(
    X_test,
    scaler)

In [12]:
# PYTORCH DATASETS
X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.float32
).view(-1, 1)

X_val_tensor = torch.tensor(
    X_val_scaled,
    dtype=torch.float32)

y_val_tensor = torch.tensor(
    y_val,
    dtype=torch.float32
).view(-1, 1)

X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
).to(DEVICE)

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True)

In [13]:
# DROPOUT ANN
class DropoutANN(nn.Module):

    def __init__(
        self,
        input_dim,
        h1,
        h2,
        dropout_rate
    ):

        super().__init__()

        self.fc1 = nn.Linear(
            input_dim,
            h1)

        self.fc2 = nn.Linear(
            h1,
            h2)

        self.out = nn.Linear(
            h2,
            1)

        # IMPORTANT:
        # Dropout is part of the architecture DURING TRAINING.
        self.dropout1 = nn.Dropout(
            p=dropout_rate)

        self.dropout2 = nn.Dropout(
            p=dropout_rate )

        self.activation = nn.ReLU()

    def forward(self, x):

        x = self.fc1(x)
        x = self.activation(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = self.activation(x)
        x = self.dropout2(x)

        x = self.out(x)

        return x

In [14]:
# CREATE MODEL FROM SCRATCH

model = DropoutANN(
    input_dim=len(FEATURE_COLS),
    h1=H1,
    h2=H2,
    dropout_rate=DROPOUT_RATE
).to(DEVICE)

print("\nModel:")
print(model)


Model:
DropoutANN(
  (fc1): Linear(in_features=6, out_features=54, bias=True)
  (fc2): Linear(in_features=54, out_features=51, bias=True)
  (out): Linear(in_features=51, out_features=1, bias=True)
  (dropout1): Dropout(p=0.2, inplace=False)
  (dropout2): Dropout(p=0.2, inplace=False)
  (activation): ReLU()
)


In [15]:
# OPTIMIZER
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE)

criterion = nn.MSELoss()

In [16]:
# VALIDATION FUNCTION

@torch.no_grad()
def validation_loss():

    model.eval()

    Xv = X_val_tensor.to(DEVICE)
    yv = y_val_tensor.to(DEVICE)

    pred = model(Xv)

    loss = criterion(
        pred,
        yv)
    return float(loss.item())

In [17]:
# TRAINING
best_val_loss = float("inf")
best_state = None
patience_counter = 0

history = []

print("\nTraining dropout model from scratch...")

for epoch in range(1, EPOCHS + 1):

    # Dropout ACTIVE during training
    model.train()

    running_loss = 0.0
    n_samples = 0

    for xb, yb in train_loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()

        pred = model(xb)

        loss = criterion(
            pred,
            yb
        )

        loss.backward()

        optimizer.step()

        batch_n = len(xb)

        running_loss += (
            loss.item() * batch_n
        )

        n_samples += batch_n

    train_loss = (
        running_loss / n_samples
    )

    val_loss = validation_loss()

    history.append({
        "epoch": epoch,
        "train_mse": train_loss,
        "val_mse": val_loss
    })

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }

        patience_counter = 0

    else:

        patience_counter += 1

    if epoch == 1 or epoch % 25 == 0:

        print(
            f"Epoch {epoch:4d} | "
            f"Train MSE={train_loss:.6f} | "
            f"Val MSE={val_loss:.6f}")

    if patience_counter >= PATIENCE:

        print(f"\nEarly stopping at epoch {epoch}")

        break


Training dropout model from scratch...
Epoch    1 | Train MSE=179.777832 | Val MSE=308.937134
Epoch   25 | Train MSE=26.356671 | Val MSE=17.812347
Epoch   50 | Train MSE=21.628665 | Val MSE=13.266973
Epoch   75 | Train MSE=19.907716 | Val MSE=16.132492
Epoch  100 | Train MSE=16.881038 | Val MSE=4.673790
Epoch  125 | Train MSE=16.339944 | Val MSE=6.101840
Epoch  150 | Train MSE=13.466610 | Val MSE=14.507531
Epoch  175 | Train MSE=12.778059 | Val MSE=8.821626

Early stopping at epoch 191


In [18]:
# RESTORE BEST MODEL

if best_state is not None:

    model.load_state_dict(best_state)

model.to(DEVICE)

print(f"\nBest validation MSE: "
    f"{best_val_loss:.6f}")


Best validation MSE: 1.637336


In [19]:
# SAVE TRAINING HISTORY
pd.DataFrame(history).to_csv(
    os.path.join(
        OUT_DIR,
        "mc_dropout_training_history.csv"
    ),
    index=False)

In [20]:
# STANDARD DETERMINISTIC TEST PREDICTION
model.eval()

with torch.no_grad():

    deterministic_pred = (
        model(X_test_tensor)
        .cpu()
        .numpy()
        .reshape(-1))

In [21]:
# MC DROPOUT PREDICTIONS
print(
    f"\nRunning {MC_RUNS} stochastic "
    f"MC Dropout forward passes...")

mc_predictions = []

# IMPORTANT:
# model.train() activates the dropout layers.
# No weights are changed because gradients are disabled.
with torch.no_grad():

    model.train()

    for i in range(MC_RUNS):

        pred = (
            model(X_test_tensor)
            .cpu()
            .numpy()
            .reshape(-1)
        )

        mc_predictions.append(pred)

mc_predictions = np.asarray(
    mc_predictions)

# Shape:
# [MC_RUNS, N_TEST]


Running 200 stochastic MC Dropout forward passes...


In [22]:
# MC STATISTICS
mc_mean = np.mean(
    mc_predictions,
    axis=0)

mc_std = np.std(
    mc_predictions,
    axis=0,
    ddof=1)

lower_95 = np.percentile(
    mc_predictions,
    2.5,
    axis=0)

upper_95 = np.percentile(
    mc_predictions,
    97.5,
    axis=0)

In [23]:
# 95% COVERAGE
covered = (
    (y_test >= lower_95) & (y_test <= upper_95))

coverage = np.mean(covered.astype(np.float32))

print(f"\n95% coverage: {coverage * 100:.2f}%")


95% coverage: 76.11%


In [24]:
# INTERVAL WIDTH

interval_width = (upper_95 - lower_95)
mean_interval_width = np.mean(
    interval_width)

median_interval_width = np.median( interval_width)

In [25]:
# MC MEAN METRICS
rmse = np.sqrt(
    mean_squared_error(
        y_test,
        mc_mean
    )
)

mae = mean_absolute_error(y_test,mc_mean)

r2 = r2_score(y_test, mc_mean)

print("\nMC Dropout Test Results")
print("-" * 50)

print(f"RMSE: {rmse:.6f}")

print(f"MAE : {mae:.6f}")

print(f"R²  : {r2:.6f}")

print(
    f"95% coverage: "
    f"{coverage * 100:.2f}%")

print(f"Mean interval width: " f"{mean_interval_width:.6f}")

print(f"Median interval width: "
      f"{median_interval_width:.6f}")


MC Dropout Test Results
--------------------------------------------------
RMSE: 4.976746
MAE : 1.789927
R²  : 0.870888
95% coverage: 76.11%
Mean interval width: 6.540291
Median interval width: 2.965332


In [26]:
# CP > 50% ANALYSIS
high_mask = y_test > 50

high_results = {
    "n": int(np.sum(high_mask)),
    "RMSE": None,
    "MAE": None,
    "R2": None,
    "coverage_95": None
}

if np.sum(high_mask) >= 2:

    y_high = y_test[high_mask]

    pred_high = mc_mean[high_mask]

    high_results["RMSE"] = float(
        np.sqrt(
            mean_squared_error(
                y_high,
                pred_high
            )
        )
    )

    high_results["MAE"] = float(
        mean_absolute_error(
            y_high,
            pred_high
        )
    )

    high_results["R2"] = float(
        r2_score(
            y_high,
            pred_high
        )
    )

    high_results["coverage_95"] = float(
        np.mean(
            covered[high_mask]
        )
    )

print("\nCP > 50%")
print("-" * 50)

print(
    f"N: {high_results['n']}"
)

if high_results["RMSE"] is not None:

    print(
        f"RMSE: "
        f"{high_results['RMSE']:.6f}"
    )

    print(
        f"MAE: "
        f"{high_results['MAE']:.6f}"
    )

    print(
        f"R²: "
        f"{high_results['R2']:.6f}"
    )

    print(
        f"95% coverage: "
        f"{high_results['coverage_95'] * 100:.2f}%"
    )


CP > 50%
--------------------------------------------------
N: 6
RMSE: 6.498451
MAE: 4.925402
R²: 0.266276
95% coverage: 100.00%


In [27]:
# SAVE INTERVAL DATA
interval_df = pd.DataFrame({

    "Observed_CP": y_test,

    "MC_Mean": mc_mean,

    "MC_Std": mc_std,

    "Lower_95": lower_95,

    "Upper_95": upper_95,

    "Interval_Width": interval_width,

    "Covered_95": covered

})

interval_df.to_csv(
    os.path.join(
        OUT_DIR,
        "mc_dropout_test_intervals.csv"
    ),
    index=False
)

In [28]:
# SAVE SUMMARY
summary = {

    "model": "MC Dropout ANN",

    "seed": SEED,

    "architecture": {
        "input_features": len(FEATURE_COLS),
        "H1": H1,
        "H2": H2
    },

    "learning_rate": LEARNING_RATE,

    "dropout_rate": DROPOUT_RATE,

    "mc_runs": MC_RUNS,

    "total_observations": int(
        len(df)
    ),

    "development_n": int(
        len(X_dev)
    ),

    "test_n": int(
        len(X_test)
    ),

    "preprocessing": {
        "suction_transform": "log1p",
        "scaler": "StandardScaler",
        "scaler_fit_on": "development_training_subset_only"
    },

    "test_metrics": {
        "RMSE": float(rmse),
        "MAE": float(mae),
        "R2": float(r2)
    },

    "uncertainty": {
        "interval": "empirical 95% MC Dropout interval",
        "coverage": float(coverage),
        "coverage_percent": float(
            coverage * 100
        ),
        "mean_interval_width": float(
            mean_interval_width
        ),
        "median_interval_width": float(
            median_interval_width
        )
    },

    "CP_greater_than_50": high_results
}

with open(
    os.path.join(
        OUT_DIR,
        "mc_dropout_results.json"
    ),
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2    )

In [29]:
# SAVE MODEL
torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "H1": H1,
        "H2": H2,

        "learning_rate":
            LEARNING_RATE,

        "dropout_rate":
            DROPOUT_RATE,

        "feature_cols":
            FEATURE_COLS,

        "target_col":
            TARGET_COL,

        "seed":
            SEED
    },

    os.path.join(
        OUT_DIR,
        "mc_dropout_model.pth"
    )
)

print("\nSaved files:")
print(
    os.path.join(
        OUT_DIR,
        "mc_dropout_test_intervals.csv"
    )
)

print(
    os.path.join(
        OUT_DIR,
        "mc_dropout_results.json"
    )
)

print(os.path.join(OUT_DIR,"mc_dropout_training_history.csv"))

print(
    os.path.join(
        OUT_DIR,
        "mc_dropout_model.pth"
    ))
print("\nDone.")


Saved files:
/content/drive/MyDrive/NNsGA/mc_dropout_results/mc_dropout_test_intervals.csv
/content/drive/MyDrive/NNsGA/mc_dropout_results/mc_dropout_results.json
/content/drive/MyDrive/NNsGA/mc_dropout_results/mc_dropout_training_history.csv
/content/drive/MyDrive/NNsGA/mc_dropout_results/mc_dropout_model.pth

Done.
